# S04 J1 — Databricks ingestion patterns

Objectif : comparer trois stratégies d'ingestion :
1. COPY INTO
2. Auto Loader
3. Incrémental par curseur / watermark






## Pattern 1 — COPY INTO

In [0]:
%sql
create schema if not exists workspace.study_s04_ingestion;

In [0]:
%sql
create volume if not exists workspace.study_s04_ingestion.landing;

In [0]:
%sql
show volumes in workspace.study_s04_ingestion;

database,volume_name
study_s04_ingestion,landing


In [0]:
from datetime import datetime

batch_data = [
    (1, datetime(2026, 8, 30, 18, 0), "A", 10.5, datetime(2026, 8, 30, 18, 1)),
    (2, datetime(2026, 8, 30, 18, 5), "B", 20.0, datetime(2026, 8, 30, 18, 6)),
    (3, datetime(2026, 8, 30, 18, 10), "A", 15.0, datetime(2026, 8, 30, 18, 11)),
]

columns = [
    "event_id",
    "event_time",
    "category",
    "amount",
    "updated_at"
]

batch_df = spark.createDataFrame(batch_data, columns)

batch_df.display()

event_id,event_time,category,amount,updated_at
1,2026-08-30T18:00:00.000Z,A,10.5,2026-08-30T18:01:00.000Z
2,2026-08-30T18:05:00.000Z,B,20.0,2026-08-30T18:06:00.000Z
3,2026-08-30T18:10:00.000Z,A,15.0,2026-08-30T18:11:00.000Z


In [0]:
batch_df.write \
    .mode("overwrite") \
    .json("/Volumes/workspace/study_s04_ingestion/landing/batch")

In [0]:
%sql
create table if not exists workspace.study_s04_ingestion.copy_into_events(
    event_id bigint,
    event_time timestamp,
    category string,
    amount double,
    updated_at timestamp
)
using delta


In [0]:
%sql

COPY INTO workspace.study_s04_ingestion.copy_into_events
FROM (
    SELECT
        CAST(event_id AS BIGINT) AS event_id,
        CAST(event_time AS TIMESTAMP) AS event_time,
        category,
        CAST(amount AS DOUBLE) AS amount,
        CAST(updated_at AS TIMESTAMP) AS updated_at
    FROM '/Volumes/workspace/study_s04_ingestion/landing/batch'
)
FILEFORMAT = JSON

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
3,3,0


In [0]:
source_df = spark.read.json("/Volumes/workspace/study_s04_ingestion/landing/batch")
source_df.printSchema()

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- event_id: long (nullable = true)
 |-- event_time: string (nullable = true)
 |-- updated_at: string (nullable = true)



In [0]:
%sql
SELECT *
FROM workspace.study_s04_ingestion.copy_into_events
ORDER BY event_id;


event_id,event_time,category,amount,updated_at
1,2026-08-30T18:00:00.000Z,A,10.5,2026-08-30T18:01:00.000Z
2,2026-08-30T18:05:00.000Z,B,20.0,2026-08-30T18:06:00.000Z
3,2026-08-30T18:10:00.000Z,A,15.0,2026-08-30T18:11:00.000Z


In [0]:
%sql
SELECT COUNT(*) AS nb_lignes
FROM workspace.study_s04_ingestion.copy_into_events;

nb_lignes
3


In [0]:
from datetime import datetime

batch_data_2 = [
    (4, datetime(2026, 8, 30, 18, 15), "C", 30.0, datetime(2026, 8, 30, 18, 16)),
    (5, datetime(2026, 8, 30, 18, 20), "B", 12.5, datetime(2026, 8, 30, 18, 21)),
]

batch_df_2 = spark.createDataFrame(batch_data_2, columns)

batch_df_2.display()

event_id,event_time,category,amount,updated_at
4,2026-08-30T18:15:00.000Z,C,30.0,2026-08-30T18:16:00.000Z
5,2026-08-30T18:20:00.000Z,B,12.5,2026-08-30T18:21:00.000Z


In [0]:
batch_df_2.write\
    .mode('overwrite')\
    .json('/Volumes/workspace/study_s04_ingestion/landing/batch_002')

In [0]:
%sql

COPY INTO workspace.study_s04_ingestion.copy_into_events
FROM (
    SELECT
        CAST(event_id AS BIGINT) AS event_id,
        CAST(event_time AS TIMESTAMP) AS event_time,
        category,
        CAST(amount AS DOUBLE) AS amount,
        CAST(updated_at AS TIMESTAMP) AS updated_at
    FROM '/Volumes/workspace/study_s04_ingestion/landing/batch_002'
)
FILEFORMAT = JSON

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
2,2,0


In [0]:
%sql
SELECT COUNT(*) AS nb_lignes
FROM workspace.study_s04_ingestion.copy_into_events;

nb_lignes
5



## Pattern 2 — Auto Loader

In [0]:
stream_data_1 = [
    (101, datetime(2026,8,30,19,0),'A',100.0,datetime(2026,8,30,19,1)),
    (102, datetime(2026,8,30,19,5),'B',200.0,datetime(2026,8,30,19,6)),
]

stream_df_1 = spark.createDataFrame(stream_data_1, columns)

stream_df_1.display()

event_id,event_time,category,amount,updated_at
101,2026-08-30T19:00:00.000Z,A,100.0,2026-08-30T19:01:00.000Z
102,2026-08-30T19:05:00.000Z,B,200.0,2026-08-30T19:06:00.000Z


In [0]:
stream_df_1.write\
    .mode('overwrite')\
    .json('/Volumes/workspace/study_s04_ingestion/landing/stream')

In [0]:
stream_source = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .schema(stream_df_1.schema)
        .load("/Volumes/workspace/study_s04_ingestion/landing/stream")
)

In [0]:
query = (
    stream_source.writeStream
        .option(
            'checkpointLocation',
            '/Volumes/workspace/study_s04_ingestion/landing/_checkpoints/stream'
        )
        .trigger(availableNow=True)
        .toTable('workspace.study_s04_ingestion.autoloader_events')
        )

In [0]:
query.awaitTermination()

In [0]:
%sql
SELECT *
FROM workspace.study_s04_ingestion.autoloader_events
ORDER BY event_id;

event_id,event_time,category,amount,updated_at
101,2026-08-30T19:00:00.000Z,A,100.0,2026-08-30T19:01:00.000Z
102,2026-08-30T19:05:00.000Z,B,200.0,2026-08-30T19:06:00.000Z


In [0]:
stream_data_2 = [
    (103, datetime(2026, 8, 30, 19, 10), "C", 300.0, datetime(2026, 8, 30, 19, 11)),
    (104, datetime(2026, 8, 30, 19, 15), "A", 150.0, datetime(2026, 8, 30, 19, 16)),
]

stream_df_2 = spark.createDataFrame(stream_data_2, columns)

stream_df_2.display()

event_id,event_time,category,amount,updated_at
103,2026-08-30T19:10:00.000Z,C,300.0,2026-08-30T19:11:00.000Z
104,2026-08-30T19:15:00.000Z,A,150.0,2026-08-30T19:16:00.000Z


In [0]:
stream_df_2.write \
    .mode("overwrite") \
    .json("/Volumes/workspace/study_s04_ingestion/landing/stream/stream_002")

In [0]:
stream_source = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .schema(stream_df_1.schema)
        .load("/Volumes/workspace/study_s04_ingestion/landing/stream")
)

query = (
    stream_source.writeStream
        .option(
            "checkpointLocation",
            "/Volumes/workspace/study_s04_ingestion/landing/_checkpoints/events"
        )
        .trigger(availableNow=True)
        .toTable("workspace.study_s04_ingestion.autoloader_events")
)

query.awaitTermination()

In [0]:
%sql
SELECT COUNT(*) AS nb_lignes
FROM workspace.study_s04_ingestion.autoloader_events;

nb_lignes
6


## Pattern 3 — Incrémental par curseur

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.study_s04_ingestion.source_orders (
    order_id BIGINT,
    customer STRING,
    amount DOUBLE,
    updated_at TIMESTAMP
)
USING DELTA;

In [0]:
%sql

INSERT INTO workspace.study_s04_ingestion.source_orders
VALUES
    (1, 'Alice', 100.0, TIMESTAMP '2026-08-30 20:00:00'),
    (2, 'Bob',   200.0, TIMESTAMP '2026-08-30 20:05:00'),
    (3, 'Chloe', 150.0, TIMESTAMP '2026-08-30 20:10:00');

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql

SELECT *
FROM workspace.study_s04_ingestion.source_orders
ORDER BY order_id;

order_id,customer,amount,updated_at
1,Alice,100.0,2026-08-30T20:00:00.000Z
2,Bob,200.0,2026-08-30T20:05:00.000Z
3,Chloe,150.0,2026-08-30T20:10:00.000Z


In [0]:
%sql

CREATE OR REPLACE TABLE workspace.study_s04_ingestion.incremental_orders (
    order_id BIGINT,
    customer STRING,
    amount DOUBLE,
    updated_at TIMESTAMP
)
USING DELTA;

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.study_s04_ingestion.ingestion_watermark (
    pipeline_name STRING,
    last_updated_at TIMESTAMP
)
USING DELTA;

In [0]:
%sql

INSERT INTO workspace.study_s04_ingestion.ingestion_watermark
VALUES (
    'orders_pipeline',
    TIMESTAMP '1970-01-01 00:00:00'
);

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql

SELECT *
FROM workspace.study_s04_ingestion.ingestion_watermark;

pipeline_name,last_updated_at
orders_pipeline,1970-01-01T00:00:00.000Z


In [0]:
%sql
select s.*
from workspace.study_s04_ingestion.source_orders s 
where s.updated_at > 
(
    select last_updated_at
    from workspace.study_s04_ingestion.ingestion_watermark
    where pipeline_name = 'orders_pipeline'
)
order by s.order_id

order_id,customer,amount,updated_at
1,Alice,100.0,2026-08-30T20:00:00.000Z
2,Bob,200.0,2026-08-30T20:05:00.000Z
3,Chloe,150.0,2026-08-30T20:10:00.000Z


In [0]:
%sql

merge into workspace.study_s04_ingestion.incremental_orders as target

using (
    select s.*
    from workspace.study_s04_ingestion.source_orders s 
    where s.updated_at > 
    (
        select last_updated_at
        from workspace.study_s04_ingestion.ingestion_watermark
        where pipeline_name = 'orders_pipeline'
    )
    order by s.order_id
) as source

on target.order_id = source.order_id

when matched then
    update set
    target.customer = source.customer,
    target.amount = source.amount,
    target.updated_at = source.updated_at

when not matched then
    insert (
        order_id,
        customer,
        amount,
        updated_at
    ) values (
        source.order_id,
        source.customer,
        source.amount,
        source.updated_at
    );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,0,0,3


In [0]:
%sql

SELECT *
FROM workspace.study_s04_ingestion.incremental_orders
ORDER BY order_id;

order_id,customer,amount,updated_at
1,Alice,100.0,2026-08-30T20:00:00.000Z
2,Bob,200.0,2026-08-30T20:05:00.000Z
3,Chloe,150.0,2026-08-30T20:10:00.000Z


In [0]:
%sql

update workspace.study_s04_ingestion.ingestion_watermark
set last_updated_at = (
    select max(updated_at)
    from workspace.study_s04_ingestion.source_orders
)
where pipeline_name = 'orders_pipeline'

num_affected_rows
1


In [0]:
%sql
SELECT *
FROM workspace.study_s04_ingestion.ingestion_watermark;

pipeline_name,last_updated_at
orders_pipeline,2026-08-30T20:10:00.000Z


In [0]:
%sql

SELECT s.*
FROM workspace.study_s04_ingestion.source_orders s
WHERE s.updated_at >
(
    SELECT last_updated_at
    FROM workspace.study_s04_ingestion.ingestion_watermark
    WHERE pipeline_name = 'orders_pipeline'
);

order_id,customer,amount,updated_at
